# Illanes_emilio_Lab13

**Nombre:** Emilio Alejandro Illanes Loyola  
**RUT:** 21.017.984-4  
**Fecha:** 17-06-2026


## Ejercicio 1

In [132]:
import sqlite3

with sqlite3.connect('clima_lab.db') as conn:
    cursor=conn.cursor()

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS registros(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        ciudad TEXT NOT NULL,
        temp_c REAL,
        humedad INTEGER,
        precip_mm REAL,
        viento_kmh REAL,
        fecha TEXT
    )
    ''')

    print("Tabla 'registros' creada.")

    cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name='registros'")
    print(cursor.fetchone()[0])

    cursor.execute("PRAGMA table_info(registros)")
    for col in cursor.fetchall():
        print(col[1], col[2])


Tabla 'registros' creada.
CREATE TABLE registros(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        ciudad TEXT NOT NULL,
        temp_c REAL,
        humedad INTEGER,
        precip_mm REAL,
        viento_kmh REAL,
        fecha TEXT
    )
id INTEGER
ciudad TEXT
temp_c REAL
humedad INTEGER
precip_mm REAL
viento_kmh REAL
fecha TEXT


## Ejercicio 2

In [133]:
import sqlite3

with sqlite3.connect('clima_lab.db') as conn:
    cursor=conn.cursor()

    cursor.execute('''
    INSERT INTO registros
    (ciudad,temp_c,humedad,precip_mm,viento_kmh,fecha)
    VALUES (?,?,?,?,?,?)
    ''',('Santiago',18.5,72,1.2,15.3,'2026-06-15'))

    print("ID:",cursor.lastrowid)

    datos=[
    ('Valparaíso',15.2,80,3.5,22.1,'2026-06-15'),
    ('Concepción',12.8,85,8.0,18.7,'2026-06-15'),
    ('Temuco',9.3,90,12.5,25.0,'2026-06-15'),
    ('La Serena',17.8,55,0.0,12.4,'2026-06-15')
    ]

    cursor.executemany('''
    INSERT INTO registros
    (ciudad,temp_c,humedad,precip_mm,viento_kmh,fecha)
    VALUES (?,?,?,?,?,?)
    ''',datos)

    conn.commit()

    cursor.execute("SELECT COUNT(*) FROM registros")
    print("Total registros:",cursor.fetchone()[0])


ID: 1
Total registros: 5


## Ejercicio 3

In [134]:
import sqlite3

with sqlite3.connect('clima_lab.db') as conn:
    conn.row_factory=sqlite3.Row
    cursor=conn.cursor()

    print("Listado completo")
    cursor.execute("SELECT ciudad,temp_c,fecha FROM registros")
    for fila in cursor.fetchall():
        print(f"{fila['ciudad']:12} {fila['temp_c']} {fila['fecha']}")

    print("\nCiudad más cálida")
    cursor.execute('SELECT ciudad,temp_c FROM registros ORDER BY temp_c DESC LIMIT 1')
    print(dict(cursor.fetchone()))

    print("\nTemperaturas menores a 13°C")
    cursor.execute('SELECT ciudad,temp_c FROM registros WHERE temp_c < ?',(13,))
    for fila in cursor.fetchall():
        print(dict(fila))

    print("\nEstadísticas")
    cursor.execute('''
    SELECT ciudad,
           AVG(temp_c) promedio_temp,
           MIN(humedad) humedad_min,
           MAX(humedad) humedad_max
    FROM registros
    GROUP BY ciudad
    ORDER BY promedio_temp DESC
    ''')
    for fila in cursor.fetchall():
        print(dict(fila))


Listado completo
Santiago     18.5 2026-06-15
Valparaíso   15.2 2026-06-15
Concepción   12.8 2026-06-15
Temuco       9.3 2026-06-15
La Serena    17.8 2026-06-15

Ciudad más cálida
{'ciudad': 'Santiago', 'temp_c': 18.5}

Temperaturas menores a 13°C
{'ciudad': 'Concepción', 'temp_c': 12.8}
{'ciudad': 'Temuco', 'temp_c': 9.3}

Estadísticas
{'ciudad': 'Santiago', 'promedio_temp': 18.5, 'humedad_min': 72, 'humedad_max': 72}
{'ciudad': 'La Serena', 'promedio_temp': 17.8, 'humedad_min': 55, 'humedad_max': 55}
{'ciudad': 'Valparaíso', 'promedio_temp': 15.2, 'humedad_min': 80, 'humedad_max': 80}
{'ciudad': 'Concepción', 'promedio_temp': 12.8, 'humedad_min': 85, 'humedad_max': 85}
{'ciudad': 'Temuco', 'promedio_temp': 9.3, 'humedad_min': 90, 'humedad_max': 90}


## Ejercicio 4

In [135]:
import sqlite3

with sqlite3.connect('clima_lab.db') as conn:
    cursor=conn.cursor()

    try:
        cursor.execute(
            'UPDATE registros SET temp_c=? WHERE ciudad=?',
            (16.0,'Valparaíso')
        )
        print("UPDATE 1:",cursor.rowcount)

        cursor.execute(
            'UPDATE registros SET humedad=humedad+5 WHERE temp_c < ?',
            (10,)
        )
        print("UPDATE 2:",cursor.rowcount)

        cursor.execute(
            'DELETE FROM registros WHERE ciudad=?',
            ('Temuco',)
        )
        print("DELETE:",cursor.rowcount)

        cursor.execute("SELECT COUNT(*) FROM registros WHERE ciudad='Temuco'")
        print("Temuco restantes:",cursor.fetchone()[0])

        conn.commit()

    except sqlite3.Error as e:
        print(e)
        conn.rollback()


UPDATE 1: 1
UPDATE 2: 1
DELETE: 1
Temuco restantes: 0


## Ejercicio 5

In [136]:
import pandas as pd
import sqlite3

df=pd.read_csv('S15_registros_climaticos.csv')

print("INFO")
df.info()

print("\nNulos")
print(df.isnull().sum())

filas_originales=len(df)

df['ciudad']=df['ciudad'].str.title()

df['fecha']=pd.to_datetime(
    df['fecha'],
    errors='coerce',
    dayfirst=True
).dt.strftime('%Y-%m-%d')

df=df.drop_duplicates(subset=['ciudad','fecha'])

df=df[(df['temp_c']>=-5) & (df['temp_c']<=50)]

filas_limpias=len(df)

con=sqlite3.connect('clima_lab.db')

df.to_sql(
    'registros_csv',
    con,
    if_exists='replace',
    index=False
)

q1=pd.read_sql_query('''
SELECT ciudad,AVG(temp_c) promedio_temp
FROM registros_csv
GROUP BY ciudad
ORDER BY promedio_temp DESC
LIMIT 1
''',con)

q2=pd.read_sql_query('''
SELECT ciudad,
AVG(precip_mm) promedio_precipitacion
FROM registros_csv
WHERE fecha BETWEEN '2026-01-01' AND '2026-03-31'
GROUP BY ciudad
ORDER BY promedio_precipitacion DESC
''',con)

q3=pd.read_sql_query('''
SELECT fecha,ciudad,temp_c
FROM registros_csv
ORDER BY temp_c DESC
LIMIT 5
''',con)

print("\nFilas originales:",filas_originales)
print("Filas limpias:",filas_limpias)
print("Filas eliminadas:",filas_originales-filas_limpias)

print("\nCiudad con mayor temperatura promedio")
print(q1)

print("\nPromedio precipitación enero-marzo 2026")
print(q2)

print("\nTop 5 temperaturas")
print(q3)

con.close()


INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ciudad      208 non-null    object 
 1   temp_c      194 non-null    float64
 2   humedad     198 non-null    float64
 3   precip_mm   208 non-null    float64
 4   viento_kmh  208 non-null    float64
 5   fecha       208 non-null    object 
dtypes: float64(4), object(2)
memory usage: 9.9+ KB

Nulos
ciudad         0
temp_c        14
humedad       10
precip_mm      0
viento_kmh     0
fecha          0
dtype: int64

Filas originales: 208
Filas limpias: 176
Filas eliminadas: 32

Ciudad con mayor temperatura promedio
        ciudad  promedio_temp
0  Antofagasta      20.673333

Promedio precipitación enero-marzo 2026
         ciudad  promedio_precipitacion
0      Santiago                7.125000
1      Rancagua                3.620000
2     La Serena                3.514286
3        Temuco           

/tmp/ipykernel_238/214137855.py:16: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['fecha']=pd.to_datetime(
